# Landscape Classification & AI-Generated Image Detection

**Course:** Machine Learning (Graduate Level)  
**Author:** Amirreza Taheri  
**Dataset:** Real vs. AI-Generated Landscape Benchmark (`jungle`, `mountain`, `sea`)  

---

## Executive Summary
This project investigates the dual classification problem on natural landscape imagery:
1. **Multi-Class Scene Recognition:** Classifying images into `jungle`, `mountain`, or `sea` (3 classes).
2. **Synthetic Authenticity Detection:** Discriminating between authentic field photographs and synthetic scenes generated by text-to-image diffusion models (Stable Diffusion, DALL-E) (binary classification: `real` vs. `fake`).

Two core computational methodologies are developed and comparatively benchmarked:
- **Custom Deep CNN:** Trained from scratch using an 8-layer convolutional architecture with batch normalization and global average pooling.
- **Transfer Learning:** Fine-tuning pre-trained deep residual and mobile architectures (**ResNet-152V2** and **MobileNetV3-Large**).


## 1. Environment Configuration & Dependencies


In [ ]:
import os
import re
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet152V2, MobileNetV3Large
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set deterministic seed
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Verify GPU acceleration
physical_devices = tf.config.list_physical_devices('GPU')
print(f"GPUs Available: {len(physical_devices)}")
if physical_devices:
    print(f"Active Device: {physical_devices[0]}")


## 2. Dataset Ingestion & Dual-Label Parsing
Rather than duplicating raw image files on disk, we construct a unified tabular index from `train/`. Each image contains both ground-truth dimensions:
- **Landscape class** (from parent directory: `jungle`, `mountain`, `sea`)
- **Authenticity label** (from structured filename: `real` vs. `fake`)


In [ ]:
dataset_root = pathlib.Path('train')

records = []
for file_path in dataset_root.rglob('*'):
    if file_path.suffix.lower() in ('.jpg', '.jpeg', '.png'):
        fname = file_path.name.lower()
        landscape_dir = file_path.parent.name.lower()
        
        # Determine authenticity label
        if 'fake' in fname:
            auth_label = 'fake'
        elif 'real' in fname:
            auth_label = 'real'
        else:
            continue
            
        # Determine landscape label
        if landscape_dir in ('jungle', 'mountain', 'sea'):
            landscape_label = landscape_dir
        elif 'jungle' in fname:
            landscape_label = 'jungle'
        elif 'mountain' in fname:
            landscape_label = 'mountain'
        elif 'sea' in fname:
            landscape_label = 'sea'
        else:
            continue
            
        records.append({
            'file_path': str(file_path),
            'landscape': landscape_label,
            'authenticity': auth_label
        })

df = pd.DataFrame(records)
print(f"Total indexed images: {len(df)}")
df.head()


In [ ]:
# Verify class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='landscape', ax=axes[0], palette='viridis')
axes[0].set_title('Landscape Distribution')
axes[0].set_ylabel('Sample Count')

sns.countplot(data=df, x='authenticity', ax=axes[1], palette='magma')
axes[1].set_title('Authenticity Distribution (Real vs. Synthetic)')
axes[1].set_ylabel('Sample Count')

plt.tight_layout()
plt.show()


## 3. Stratified Partitioning & tf.data Pipelines
We perform a stratified train (70%), validation (15%), and test (15%) split to ensure balanced representation across both targets.


In [ ]:
# Create composite label for joint stratification
df['strat_key'] = df['landscape'] + '_' + df['authenticity']

train_df, test_val_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['strat_key'])
val_df, test_df = train_test_split(test_val_df, test_size=0.50, random_state=SEED, stratify=test_val_df['strat_key'])

print(f"Train samples: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation samples: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test samples: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

# Label mappings
LANDSCAPE_MAP = {'jungle': 0, 'mountain': 1, 'sea': 2}
AUTH_MAP = {'real': 0, 'fake': 1}

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(path):
    img_raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_raw, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img

def create_dataset(dataframe, target_col, label_map, is_training=False):
    paths = dataframe['file_path'].values
    labels = dataframe[target_col].map(label_map).values
    
    path_ds = tf.data.Dataset.from_tensor_slices(paths)
    image_ds = path_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)
    
    ds = tf.data.Dataset.zip((image_ds, label_ds))
    if is_training:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
    return ds

# Datasets for Task A: Landscape
train_ds_landscape = create_dataset(train_df, 'landscape', LANDSCAPE_MAP, is_training=True)
val_ds_landscape = create_dataset(val_df, 'landscape', LANDSCAPE_MAP, is_training=False)
test_ds_landscape = create_dataset(test_df, 'landscape', LANDSCAPE_MAP, is_training=False)

# Datasets for Task B: Authenticity
train_ds_auth = create_dataset(train_df, 'authenticity', AUTH_MAP, is_training=True)
val_ds_auth = create_dataset(val_df, 'authenticity', AUTH_MAP, is_training=False)
test_ds_auth = create_dataset(test_df, 'authenticity', AUTH_MAP, is_training=False)


## 4. Custom Deep CNN Architecture (From Scratch)
Implementing the 8-stage convolutional network with batch normalization, variable kernel strides, and global average pooling.


In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.1, seed=SEED),
    layers.RandomZoom(0.1, seed=SEED),
], name="data_augmentation")

def build_scratch_cnn(num_classes, model_name="scratch_cnn"):
    inputs = layers.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255.0)(x)
    
    # Layer 1: 5x5 Conv, 64 filters, stride 2 + MaxPool
    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    
    # Layer 2: 3x3 Conv, 64 filters
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Layers 3-5: 3x3 Conv, 128 filters
    for _ in range(3):
        x = layers.Conv2D(128, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
    # Layer 6: 3x3 Conv, 256 filters, stride 4
    x = layers.Conv2D(256, (3, 3), strides=(4, 4), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Layer 7: 3x3 Conv, 256 filters
    x = layers.Conv2D(256, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Layer 8: 3x3 Conv, 512 filters, stride 2
    x = layers.Conv2D(512, (3, 3), strides=(2, 2), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Classifier Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name="predictions")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=model_name)
    return model

scratch_landscape_model = build_scratch_cnn(num_classes=3, model_name="scratch_cnn_landscape")
scratch_landscape_model.summary()


### Training Custom CNN on Task A (Landscape Classification)


In [ ]:
scratch_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

scratch_landscape_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_scratch_landscape = scratch_landscape_model.fit(
    train_ds_landscape,
    validation_data=val_ds_landscape,
    epochs=15,
    callbacks=scratch_callbacks
)


### Training Custom CNN on Task B (Real vs. Synthetic Classification)


In [ ]:
scratch_auth_model = build_scratch_cnn(num_classes=2, model_name="scratch_cnn_authenticity")
scratch_auth_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_scratch_auth = scratch_auth_model.fit(
    train_ds_auth,
    validation_data=val_ds_auth,
    epochs=15,
    callbacks=scratch_callbacks
)


## 5. Transfer Learning (ResNet-152V2 & MobileNetV3-Large)
We leverage pre-trained weights from ImageNet-1k, utilizing frozen feature extractors followed by fine-tuning.


In [ ]:
def build_transfer_model(base_model_fn, num_classes, model_name):
    inputs = layers.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255.0)(x)
    
    base_model = base_model_fn(weights='imagenet', include_top=False, input_tensor=x)
    base_model.trainable = False  # Freeze initial feature extractor
    
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name=model_name)
    return model, base_model

# Task A: Landscape Models
resnet_landscape, base_resnet = build_transfer_model(ResNet152V2, 3, "resnet152_landscape")
mobilenet_landscape, base_mobilenet = build_transfer_model(MobileNetV3Large, 3, "mobilenet_landscape")

# Compile transfer models
resnet_landscape.compile(optimizer=optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mobilenet_landscape.compile(optimizer=optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
transfer_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

print("Training ResNet-152V2 on Landscape Classification...")
history_resnet_landscape = resnet_landscape.fit(
    train_ds_landscape,
    validation_data=val_ds_landscape,
    epochs=10,
    callbacks=transfer_callbacks
)

print("Training MobileNetV3-Large on Landscape Classification...")
history_mobilenet_landscape = mobilenet_landscape.fit(
    train_ds_landscape,
    validation_data=val_ds_landscape,
    epochs=10,
    callbacks=transfer_callbacks
)


## 6. Empirical Evaluation & Comparative Benchmarks
Computing full classification metrics on the unseen test partition.


In [ ]:
def evaluate_model_performance(model, test_ds, class_names, title="Model Evaluation"):
    y_true = []
    y_pred_probs = []
    
    for images, labels in test_ds:
        probs = model.predict(images, verbose=0)
        y_pred_probs.append(probs)
        y_true.extend(labels.numpy())
        
    y_pred_probs = np.vstack(y_pred_probs)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # Accuracy & Report
    acc = accuracy_score(y_true, y_pred)
    print(f"=== {title} ===")
    print(f"Test Accuracy: {acc * 100:.2f}%
")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{title} - Confusion Matrix")
    plt.xlabel('Predicted Label')
    plt.ylabel('Ground Truth Label')
    plt.show()
    
    return acc


In [ ]:
# Evaluate Landscape Models
landscape_classes = ['jungle', 'mountain', 'sea']
acc_scratch_ls = evaluate_model_performance(scratch_landscape_model, test_ds_landscape, landscape_classes, "Scratch CNN (Landscape)")
acc_resnet_ls = evaluate_model_performance(resnet_landscape, test_ds_landscape, landscape_classes, "ResNet-152V2 (Landscape)")
acc_mobilenet_ls = evaluate_model_performance(mobilenet_landscape, test_ds_landscape, landscape_classes, "MobileNetV3 (Landscape)")


In [ ]:
# Summary Performance Table
results_df = pd.DataFrame([
    {"Model": "Custom CNN (From Scratch)", "Task": "Landscape (3-class)", "Test Accuracy (%)": f"{acc_scratch_ls*100:.2f}"},
    {"Model": "ResNet-152V2 (Transfer Learning)", "Task": "Landscape (3-class)", "Test Accuracy (%)": f"{acc_resnet_ls*100:.2f}"},
    {"Model": "MobileNetV3-Large (Transfer Learning)", "Task": "Landscape (3-class)", "Test Accuracy (%)": f"{acc_mobilenet_ls*100:.2f}"},
])
display(results_df)
